# 模块 4 节点壳（`/detector`）—— 消息转换与可视化

演示 `scripts/detector_node.py` 中**不依赖 ROS master 的纯函数**：

- `detections_to_msg(detections)` —— `list[DetectedTarget]` → `uav_tracking/Detections`
- `draw_detections(image, detections)` —— 方案书要求的 bbox 绘制
- `image_msg_to_array(msg, bridge)` —— `sensor_msgs/Image` → BGR `np.ndarray`

> **运行前提**：已 `source /opt/ros/noetic/setup.bash` 与
> `source catkin_ws/devel/setup.bash`（需能 import `rospy` 与生成的 `uav_tracking.msg`）。
> 未 source 时下方单元格会给出明确提示。

## 0. 环境准备

In [ ]:
import os, sys, importlib

REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))   # ai4/
SCRIPTS = os.path.join(REPO, "catkin_ws", "src", "uav_tracking", "scripts")
LIB = os.path.join(REPO, "catkin_ws", "src", "uav_tracking", "lib")
for p in (os.path.join(REPO, "code"), SCRIPTS, LIB):
    if p not in sys.path:
        sys.path.insert(0, p)
print("REPO     =", REPO)

try:
    import rospy
    print("rospy    OK")
except ImportError as exc:
    print("[阻塞] 未能 import rospy，请先 source ROS 环境：", exc)
    print("       source /opt/ros/noetic/setup.bash && source catkin_ws/devel/setup.bash")

## 1. `detections_to_msg` —— 含「空也发」

**关键契约**：无目标时必须产出**合法**的空消息（`targets` 为空数组），
不得静默——`/state_machine` 的帧计数依赖消息到达节拍。

In [ ]:
try:
    import detector_node as dn
    from datatypes import DetectedTarget
    from uav_tracking.msg import Detections

    # --- 空列表：必须仍是合法消息 ---
    empty = dn.detections_to_msg([])
    print("空列表   ->", type(empty).__name__, " targets =", len(empty.targets))

    # --- 有目标 ---
    msg = dn.detections_to_msg([
        DetectedTarget(class_id=0, confidence=0.91, bbox=(280.0, 200.0, 120.0, 90.0)),
    ])
    t = msg.targets[0]
    print("有目标   -> targets =", len(msg.targets))
    print("            class_id=%d confidence=%.2f bbox=%s"
          % (t.class_id, t.confidence, [round(float(v), 1) for v in t.bbox]))
except ImportError as exc:
    print("[跳过] 需先 source ROS 并 catkin_make：", exc)

## 2. `draw_detections` —— 方案书要求的可视化交付物

要求：**不修改入参图像**（内部 copy），空检测时返回原图副本（便于确认"确实没目标"）。

In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt

    image = np.full((240, 320, 3), 100, dtype=np.uint8)
    before = image.copy()

    canvas_empty = dn.draw_detections(image, [])
    canvas_box = dn.draw_detections(image, [
        DetectedTarget(class_id=0, confidence=0.88, bbox=(120.0, 90.0, 80.0, 60.0)),
    ])
    print("入参未被修改:", bool((image == before).all()))
    print("空检测返回原图副本:", bool((canvas_empty == image).all()))

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    for ax, im, title in ((axes[0], canvas_empty, "no target"),
                          (axes[1], canvas_box, "1 target")):
        ax.imshow(im[:, :, ::-1]); ax.set_title(title)
        ax.set_xlabel("x [px]"); ax.set_ylabel("y [px]")
    plt.tight_layout(); plt.show()
except ImportError as exc:
    print("[跳过] 需先 source ROS 并 catkin_make：", exc)

## 3. `image_msg_to_array` —— 编码归一化

ROS 图像的 `encoding` 可能是 `bgr8` / `rgb8` / `mono8`。
统一转成 **3 通道 BGR**，使 `detector.detect()` 不必关心上游编码。

In [ ]:
try:
    import numpy as np
    from cv_bridge import CvBridge

    bridge = CvBridge()

    bgr = np.zeros((48, 64, 3), dtype=np.uint8); bgr[:, :, 0] = 200
    msg_bgr = bridge.cv2_to_imgmsg(bgr, encoding="bgr8")
    out = dn.image_msg_to_array(msg_bgr, bridge)
    print("bgr8  ->", out.shape, " B 通道保留在 index 0:", int(out[0, 0, 0]) == 200)

    mono = np.full((30, 40), 128, dtype=np.uint8)
    msg_mono = bridge.cv2_to_imgmsg(mono, encoding="mono8")
    out2 = dn.image_msg_to_array(msg_mono, bridge)
    print("mono8 ->", out2.shape, "（已归一化为 3 通道）")
except ImportError as exc:
    print("[跳过] 需先 source ROS 并 catkin_make：", exc)

## 4. 端到端 E2E

真正跑通数据流请用编排脚本（可重复执行、自动清理）：

```bash
bash test/detector/e2e_test/run_e2e.sh                       # 阶段A：链路 + 空也发
IMAGE=/path/to/target.jpg bash test/detector/e2e_test/run_e2e.sh   # 加阶段B：正常检出
```

阶段 A 断言 `/detections` 在流且 `targets` 恒为空；阶段 B 额外校验
**bbox 合法性**（宽高为正、不越界 → 专治 `xyxy`/`xywh` 搞混）。